# ML Week 15: Classification

Source: https://github.com/RafayKhattak/Digit-Classification-Pytorch/blob/main/DigitClassificationPytorch.ipynb

In [1]:
%pip install torchvision

   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ------------ --------------------------- 1.3/4.3 MB 8.6 MB/s eta 0:00:01
   ------------------------------- -------- 3.4/4.3 MB 9.2 MB/s eta 0:00:01
   ---------------------------------------- 4.3/4.3 MB 8.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Importing dependencies
import torch
from PIL import Image
from torch import nn,save,load
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [47]:
# Loading Data
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root="data", download=True, train=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [25]:
train_dataset

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train

In [32]:
import matplotlib.pyplot as plt
import numpy as np

print(train_dataset[0][1])
digit = np.array(train_dataset[0][0])


5


In [33]:
digit.shape

(28, 28)

In [34]:
# Define the image classifier model
class ImageClassifier(nn.Module):
    def __init__(self):
        super(ImageClassifier, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3),
            nn.ReLU()
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 22 * 22, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

In [50]:
# Create an instance of the image classifier model
device = torch.device("cpu")
classifier = ImageClassifier().to(device)

In [51]:
# Define the optimizer and loss function
optimizer = Adam(classifier.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

In [52]:
# Train the model
for epoch in range(10):  # Train for 10 epochs
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()  # Reset gradients
        outputs = classifier(images)  # Forward pass
        loss = loss_fn(outputs, labels)  # Compute loss
        loss.backward()  # Backward pass
        optimizer.step()  # Update weights

    print(f"Epoch:{epoch} loss is {loss.item()}")



Epoch:0 loss is 0.22961421310901642
Epoch:1 loss is 0.0005742576904594898
Epoch:2 loss is 0.07169028371572495
Epoch:3 loss is 0.002702370984479785
Epoch:4 loss is 0.0007335190894082189
Epoch:5 loss is 0.0002572373196016997
Epoch:6 loss is 0.0003071376122534275
Epoch:7 loss is 0.00615081749856472
Epoch:8 loss is 0.001481426414102316
Epoch:9 loss is 0.004794820677489042


In [ ]:
# Save the trained model
torch.save(classifier.state_dict(), 'model_state.pt')

In [ ]:
# Load the saved model
with open('model_state.pt', 'rb') as f: 
     classifier.load_state_dict(load(f))

In [ ]:
# Perform inference on an image
img = Image.open('image.jpg')
img_transform = transforms.Compose([transforms.ToTensor()])
img_tensor = img_transform(img).unsqueeze(0).to(device)
output = classifier(img_tensor)
predicted_label = torch.argmax(output)
print(f"Predicted label: {predicted_label}")

